In [1]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l1, l2
from tensorflow.keras.initializers import GlorotUniform, HeNormal, RandomNormal
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')
import random
import time

# Фиксируем seed для воспроизводимости
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

# Загрузка данных
train_df = pd.read_csv('california_housing_train.csv')
test_df = pd.read_csv('california_housing_test.csv')

features = ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
            'total_bedrooms', 'population', 'households', 'median_income']
target = 'median_house_value'

x_train_full = train_df[features].values
y_train_full = train_df[target].values
x_test = test_df[features].values
y_test = test_df[target].values

# Стандартизация
scaler = StandardScaler()
x_train_full = scaler.fit_transform(x_train_full)
x_test = scaler.transform(x_test)

# Разделение на train/val
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.2, random_state=SEED
)

In [2]:
def build_model(
    layers_config,
    activations,
    optimizer_name,
    learning_rate,
    dropout_rates,
    use_batch_norm,
    kernel_regularizer,
    bias_regularizer,
    initializer
):
    model = Sequential()
    for i, (units, act) in enumerate(zip(layers_config, activations)):
        if i == 0:
            model.add(Dense(units,
                            activation=act,
                            input_shape=(x_train.shape[1],),
                            kernel_initializer=initializer,
                            kernel_regularizer=kernel_regularizer,
                            bias_regularizer=bias_regularizer))
        else:
            model.add(Dense(units,
                            activation=act,
                            kernel_initializer=initializer,
                            kernel_regularizer=kernel_regularizer,
                            bias_regularizer=bias_regularizer))
        if use_batch_norm[i]:
            model.add(BatchNormalization())
        if dropout_rates[i] > 0:
            model.add(Dropout(dropout_rates[i]))
    model.add(Dense(1, activation='linear', kernel_initializer=initializer))
    if optimizer_name == 'adam':
        opt = Adam(learning_rate=learning_rate)
    elif optimizer_name == 'sgd':
        opt = SGD(learning_rate=learning_rate)
    elif optimizer_name == 'rmsprop':
        opt = RMSprop(learning_rate=learning_rate)
    else:
        raise ValueError("Unsupported optimizer")
    model.compile(optimizer=opt, loss='mse', metrics=['mae'])
    return model

In [3]:
import itertools

def random_architecture():
    n_layers = np.random.choice([2, 3, 4])
    layers = []
    for _ in range(n_layers):
        layers.append(np.random.choice([16, 32, 64, 128, 256]))
    return layers

def random_activation(n):
    acts = []
    for _ in range(n):
        acts.append(np.random.choice(['relu', 'elu', 'tanh', 'swish']))
    return acts

def random_dropout(n):
    return [np.random.choice([0.0, 0.1, 0.2, 0.3]) for _ in range(n)]

def random_batch_norm(n):
    return [np.random.choice([True, False]) for _ in range(n)]

def random_regularizer():
    choice = np.random.choice(['none', 'l1', 'l2'])
    if choice == 'l1':
        return l1(1e-4)
    elif choice == 'l2':
        return l2(1e-4)
    else:
        return None

def random_initializer():
    choice = np.random.choice(['glorot', 'he', 'normal'])
    if choice == 'glorot':
        return GlorotUniform(seed=SEED)
    elif choice == 'he':
        return HeNormal(seed=SEED)
    else:
        return RandomNormal(seed=SEED)

def random_optimizer():
    return np.random.choice(['adam', 'sgd', 'rmsprop'])

def random_lr():
    return np.random.choice([1e-4, 5e-4, 1e-3, 5e-3, 1e-2])

def random_batch_size():
    return np.random.choice([16, 32, 64, 128, 256])

def random_epochs():
    return np.random.choice([200])

In [ ]:
import os
import gc
import tensorflow as tf
from tensorflow.keras.backend import clear_session

results = []
N_EXPERIMENTS = 500
CHECKPOINT_EVERY = 25

best_test_mae = float('inf')
best_model_path = "best_model.keras"

early_stop = EarlyStopping(monitor='val_mae', patience=15, restore_best_weights=True)

for exp_id in range(N_EXPERIMENTS):
    print(f"Эксперимент {exp_id + 1}/{N_EXPERIMENTS}")
    
    # --- Генерация конфигурации ---
    layers = random_architecture()
    n = len(layers)
    activations = random_activation(n)
    dropout_rates = random_dropout(n)
    batch_norm_flags = random_batch_norm(n)
    kernel_reg = random_regularizer()
    bias_reg = random_regularizer()
    initializer = random_initializer()
    opt_name = random_optimizer()
    lr = random_lr()
    batch_size = random_batch_size()
    max_epochs = random_epochs()
    # --- Построение и обучение модели ---
    try:
        model = build_model(
            layers_config=layers,
            activations=activations,
            optimizer_name=opt_name,
            learning_rate=lr,
            dropout_rates=dropout_rates,
            use_batch_norm=batch_norm_flags,
            kernel_regularizer=kernel_reg,
            bias_regularizer=bias_reg,
            initializer=initializer
        )
    except Exception as e:
        print(f"Ошибка при создании модели: {e}")
        clear_session()
        gc.collect()
        continue

    try:
        history = model.fit(
            x_train, y_train,
            validation_data=(x_val, y_val),
            epochs=max_epochs,
            batch_size=batch_size,
            callbacks=[early_stop],
            verbose=0
        )
    except Exception as e:
        print(f"Ошибка при обучении модели: {e}")
        clear_session()
        gc.collect()
        continue

    actual_epochs = len(history.history['mae'])

    # --- Оценка MAE ---
    train_mae = model.evaluate(x_train, y_train, verbose=0)[1]
    val_mae = model.evaluate(x_val, y_val, verbose=0)[1]
    test_mae = model.evaluate(x_test, y_test, verbose=0)[1]

    # --- Сохранение результата ---
    results.append({
        'ID': exp_id + 1,
        'layers': str(layers),
        'activations': str(activations),
        'optimizer': opt_name,
        'learning_rate': lr,
        'batch_size': batch_size,
        'epochs': actual_epochs,
        'dropout': str(dropout_rates),
        'batch_norm': str(batch_norm_flags),
        'kernel_regularizer': str(kernel_reg),
        'bias_regularizer': str(bias_reg),
        'initializer': str(initializer.__class__.__name__),
        'train_mae': train_mae,
        'val_mae': val_mae,
        'test_mae': test_mae
    })

    # --- Обновление лучшей модели ---
    if test_mae < best_test_mae:
        best_test_mae = test_mae
        model.save(best_model_path)

    # --- Очистка памяти ---
    del model
    clear_session()
    gc.collect()

    # --- Промежуточное сохранение каждые CHECKPOINT_EVERY экспериментов ---
    if (exp_id + 1) % CHECKPOINT_EVERY == 0:
        df_partial = pd.DataFrame(results)
        df_partial.to_csv(f"experiment_results_checkpoint_{exp_id + 1}.csv", index=False)
        print(f"Промежуточные результаты до эксперимента {exp_id + 1} сохранены.")

# --- Финальное сохранение ---
df_final = pd.DataFrame(results)
df_final.to_csv("experiment_results.csv", index=False)
print("Все эксперименты завершены. Результаты сохранены в experiment_results.csv")

Эксперимент 1/500
Эксперимент 2/500
Эксперимент 3/500
Эксперимент 4/500
Эксперимент 5/500
Эксперимент 6/500
